# Library Instalation

In [ ]:
!apt-get install -y libeccodes-dev
!pip install cfgrib

## Methods

In [ ]:
import pandas as pd
import numpy as np
import cfgrib
import os
from datetime import datetime


def get_dataVariables_cfgrib(dataset):
  data_variables = []
  frozen = dataset.data_vars.dtypes
  for key, value in frozen.items():
    data_variables.append(key)
  return data_variables


def get_lastIndexNumber_cfgrib(dataset, var):
  data = dataset[var].values
  lastIndexNumber = len(data) - 1
  return lastIndexNumber


def get_hoursList_cfgrib(dataset):
  data = dataset['step'].values
  hours_values = data.astype('timedelta64[h]')
  hoursList = list(hours_values.astype('timedelta64[h]'))
  hours_numbers = [td.astype('timedelta64[h]').astype(int) for td in hoursList]
  return hours_numbers


def get_dateTimeDataFrame_cfgrib(dataset):
  time_data = dataset['time'].values

  dates = pd.DatetimeIndex(time_data)
  temp_years = dates.year.tolist()
  temp_months = dates.month.tolist()
  temp_days = dates.day.tolist()
  temp_hours = get_hoursList_cfgrib(dataset)

  timeList = []
  years = []
  months = []
  days = []
  hours = []
  prefix = ''

  for i,day in enumerate(temp_days):
    for hour in temp_hours:
      years.append(temp_years[i])
      months.append(temp_months[i])
      days.append(temp_days[i])
      hours.append(hour)

  timeList.append(years)
  timeList.append(months)
  timeList.append(days)
  timeList.append(hours)

  timeColumns = ['year', 'month', 'day', 'hour']
  dfTime = pd.DataFrame(timeList).T
  dfTime.columns = timeColumns
  return dfTime

# Pre-Processing


In [ ]:
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

In [ ]:
# File name input
raw_file_name = "52f3ce355ef47847e00d272ba15b8f9c"
raw_file_format = ".grib"

input_directory = "/content/drive/MyDrive/grib_20260327"

input_filename = raw_file_name + raw_file_format
input_filePath = os.path.join(input_directory, input_filename)

print("input_filePath: " + input_filePath)

In [ ]:
# Import with filter
import cfgrib

prefix = 'es'
prefix_numberOfPoints = -1
grib_file_path = input_filePath
if prefix_numberOfPoints != -1:
  filtered_file_name = raw_file_name + "_" + prefix + "_" + str(prefix_numberOfPoints)
  filter_option = {'numberOfPoints': prefix_numberOfPoints, 'dataType': prefix}
else:
  filtered_file_name = raw_file_name + "_" + prefix
  filter_option = { 'dataType': prefix}

dataset = cfgrib.open_dataset(grib_file_path, filter_by_keys=filter_option)
print(dataset)

# NOTES filter list
    filter_by_keys={'dataType': 'fc'}
    filter_by_keys={'dataType': 'em'}
    filter_by_keys={'dataType': 'es'}


        filter_by_keys={'numberOfPoints': 15300, 'dataType': 'fc'}
    filter_by_keys={'numberOfPoints': 3825, 'dataType': 'fc'}

In [ ]:
# Normal import
import cfgrib

grib_file_path = input_filePath
filtered_file_name = raw_file_name

dataset = cfgrib.open_dataset(grib_file_path)
print(dataset)

# Yearly Data Processing

In [ ]:
# fetch the data
data_variables = get_dataVariables_cfgrib(dataset)
lastIndex_latitude = get_lastIndexNumber_cfgrib(dataset, 'latitude')
lastIndex_longitude = get_lastIndexNumber_cfgrib(dataset, 'longitude')
print(lastIndex_latitude)
print(lastIndex_longitude)
new_dataset = []
arraydata = []

for data_var in data_variables:
  dataVar = dataset[data_var]
  for i in range(len(dataVar.values)):
    data = dataVar.values[i][lastIndex_latitude][lastIndex_longitude]
    arraydata.append(data)
  print("Processing: {}, step: {}".format(data_var, i))
  new_dataset.append(arraydata)
  arraydata = []

In [ ]:
# Get Data
import pandas as pd

df = pd.DataFrame(new_dataset)
# df = pd.DataFrame(new_dataset, column=data_variables)

# print(df.T)
df_transposed = df.T
df_transposed.columns = data_variables
print(df_transposed)

In [ ]:
# Get Date Time
from datetime import datetime
import numpy as np
import pandas as pd

time_data = dataset['time'].values

dates = pd.DatetimeIndex(time_data)
years = dates.year.tolist()
months = dates.month.tolist()
days = dates.day.tolist()
hours = dates.hour.tolist()

timeList = []

timeList.append(years)
timeList.append(months)
timeList.append(days)
timeList.append(hours)

timeColumns = ['year', 'month', 'day', 'hour']
dfTime = pd.DataFrame(timeList).T
dfTime.columns = timeColumns
print(dfTime)
print(len(years))

In [ ]:
import pandas as pd

output_directory = os.path.join(input_directory, "output_" + raw_file_name)
if not os.path.exists(output_directory):
  os.makedirs(output_directory)
output_filename = filtered_file_name
output_filePath = os.path.join(output_directory, output_filename)

print("output: " + output_filePath)

complete_df = pd.concat([dfTime, df_transposed], axis=1)

print(complete_df)

complete_df.to_csv('{0}.csv'.format(output_filePath), index=False)
complete_df.to_excel('{0}.xlsx'.format(output_filePath), index=False)

Notes

cfgrib
DatasetBuildError: multiple values for unique key, try re-open the file with one of:

```
    filter_by_keys={'numberOfPoints': 3825}
    filter_by_keys={'numberOfPoints': 14817}
```



# Monthly Data Procesing

## Data Processing Group



In [ ]:
import os
directory = "/content/drive/MyDrive/request konvert/SuhuRadiasiPresip"
data_lists = os.listdir(directory)
data_list = [file for file in data_lists if file.endswith(".grib")]
print(data_list)

filename = data_list[1]
filepath = os.path.join(directory, filename)
# print(filepath)

dataset = cfgrib.open_dataset(filepath)
print(dataset)

# print(dataset['t2m'].values[28][0][80][1120])
# print(dataset['t2m'][28][0][80][1120].values)
# print(len(dataset['t2m']))
# print(len(dataset['t2m']['time']))
# print(len(dataset['t2m']['step']))

In [ ]:
print(dataset['t2m'][28][0][80].values)


In [ ]:
import cfgrib
import pandas as pd

new_dataset = []
temp_dataset = []
temp_data = []
df_time = pd.DataFrame()
df_dataset = pd.DataFrame()

for data_grib in data_list:
  # get dataset
  filepath = os.path.join(directory, data_grib)
  dataset = cfgrib.open_dataset(filepath)
  # get parameters
  data_variables = get_dataVariables_cfgrib(dataset)
  lastIndex_latitude = get_lastIndexNumber_cfgrib(dataset, 'latitude')
  lastIndex_longitude = get_lastIndexNumber_cfgrib(dataset, 'longitude')
  for data_var in data_variables:
    dataVar = dataset[data_var]
    for t in range(len(dataVar['time'])):
      for s in range(len(dataVar['step'])):
        data = dataVar.values[t][s][lastIndex_latitude][lastIndex_longitude]
        temp_data.append(data)
        print("Processing: {}, time: {}, step: {}".format(data_grib, t, s))
    temp_dataset.append(temp_data)
    temp_data = []
  df_temp = pd.DataFrame(temp_dataset)
  dftime_temp = get_dateTimeDataFrame_cfgrib(dataset)
  df_time = pd.concat([df_time, dftime_temp], axis=0)
  df_dataset = pd.concat([df_dataset, df_temp], axis=1)
  temp_dataset = []

## Data Processing Single (Monthly)

In [ ]:
# single file

new_dataset = []
temp_dataset = []
temp_data = []
df_time = pd.DataFrame()
df_dataset = pd.DataFrame()

# get parameters
data_variables = get_dataVariables_cfgrib(dataset)
lastIndex_latitude = get_lastIndexNumber_cfgrib(dataset, 'latitude')
lastIndex_longitude = get_lastIndexNumber_cfgrib(dataset, 'longitude')
for data_var in data_variables:
  dataVar = dataset[data_var]
  for t in range(len(dataVar['time'])):
    for s in range(len(dataVar['step'])):
      data = dataVar.values[t][s][lastIndex_latitude][lastIndex_longitude]
      temp_data.append(data)
      print("Processing: {}, time: {}, step: {}".format(grib_file_path, t, s))
  temp_dataset.append(temp_data)
  temp_data = []
df_temp = pd.DataFrame(temp_dataset)
dftime_temp = get_dateTimeDataFrame_cfgrib(dataset)
df_time = pd.concat([df_time, dftime_temp], axis=0)
df_dataset = pd.concat([df_dataset, df_temp], axis=1)
temp_dataset = []

In [ ]:
# combine timedate and data
df_datasetT = df_dataset.T
df_datasetT.columns = data_variables
# print(df_datasetT)
# print(df_time)
df_complete = pd.concat([df_time, df_datasetT], axis=1)
print(df_complete)
# df_dataset = pd.concat([df_time, df_dataset], axis=1)
# print(df_dataset.T)

In [ ]:
import pandas as pd

output_directory = os.path.join(input_directory, "output_" + raw_file_name)
if not os.path.exists(output_directory):
  os.makedirs(output_directory)
output_filename = filtered_file_name
output_filePath = os.path.join(output_directory, output_filename)

print("output: " + output_filePath)

print(df_complete)

df_complete.to_csv('{0}.csv'.format(output_filePath), index=False)
df_complete.to_excel('{0}.xlsx'.format(output_filePath), index=False)

# TRIAL and ERROR

In [ ]:
new_dataset = []
temp_dataset = []
temp_data = []

data_grib = data_list[0]

# get dataset
filepath = os.path.join(directory, data_grib)
dataset = cfgrib.open_dataset(filepath)
# get parameters
data_variables = get_dataVariables_cfgrib(dataset)
lastIndex_latitude = get_lastIndexNumber_cfgrib(dataset, 'latitude')
lastIndex_longitude = get_lastIndexNumber_cfgrib(dataset, 'longitude')
for data_var in data_variables:
  dataVar = dataset[data_var]
  for t in range(len(dataVar['time'])):
    for s in range(len(dataVar['step'])):
      data = dataVar.values[t][s][lastIndex_latitude][lastIndex_longitude]
      temp_data.append(data)
      # print("Processing: {}, time: {}, step: {}".format(data_grib, t, s))
  temp_dataset.append(temp_data)
  temp_data = []

In [ ]:
# get time hour

import pandas as pd
import numpy as np


# df = pd.DataFrame(new_dataset)

# print(df)
data_grib = data_list[0]
# get dataset
filepath = os.path.join(directory, data_grib)
dataset = cfgrib.open_dataset(filepath)
print(dataset['step'].values)
timedelta_values = dataset['step'].values
hours = (timedelta_values).astype('timedelta64[h]')
# print(hours)
# print(list(hours.astype('timedelta64[h]')))




def get_hoursList_cfgrib(dataset):
  data = dataset['step'].values
  hours_values = data.astype('timedelta64[h]')
  hoursList = list(hours_values.astype('timedelta64[h]'))
  hours_numbers = [td.astype('timedelta64[h]').astype(int) for td in hoursList]
  return hours_numbers

hl = get_hoursList_cfgrib(dataset)
print(hl)

In [ ]:
# get date time


data_grib = data_list[0]
# get dataset
filepath = os.path.join(directory, data_grib)
dataset = cfgrib.open_dataset(filepath)


time_data = dataset['time'].values
# print(time_data)

dates = pd.DatetimeIndex(time_data)
temp_years = dates.year.tolist()
temp_months = dates.month.tolist()
temp_days = dates.day.tolist()
temp_hours = get_hoursList_cfgrib(dataset)

timeList = []
years = []
months = []
days = []
hours = []

for i,day in enumerate(temp_days):
  for hour in temp_hours:
    years.append(temp_years[i])
    months.append(temp_months[i])
    days.append(temp_days[i])
    hours.append(hour)

timeList.append(years)
timeList.append(months)
timeList.append(days)
timeList.append(hours)

print(timeList)
print(len(timeList[0]))

timeColumns = ['year', 'month', 'day', 'hour']
dfTime = pd.DataFrame(timeList).T
dfTime.columns = timeColumns
print(dfTime)

# Sample Data

In [ ]:
import cfgrib

grib_file_path = 'data.grib'

ds = cfgrib.open_dataset(grib_file_path)
print(ds)
temperature = ds['latitude']

print(temperature)

In [ ]:
# test get the data
datau10 = ds['u10']
print(len(datau10.values))
print(datau10.values[7295][32][448]) # this is the last data